# Enclave Evaluation — End-to-End (real Google Drive, single notebook)

Drives the entire enclave flow from one notebook:

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `SYFT_ENCLAVE_EMAIL` | Trusted execution environment |
| **Model owner** | `MODEL_OWNER_EMAIL` | Owns the model weights (NanoLM) |
| **Benchmark owner** | `BENCHMARK_OWNER_EMAIL` | Owns the demographic evaluation benchmark |
| **Researcher** | `RESEARCHER_EMAIL` | Writes the inference code, submits the job |



## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds
from syft_enclaves.settings import EnclaveSettings
from syft_job.install_source import get_syft_client_install_source
import syft_client as sc

os.environ["SYFT_CLIENT_INSTALL_SOURCE"] = f"syft-client=={sc.__version__}"
get_syft_client_install_source.cache_clear()

In [ ]:
# Load .env from this folder.
ENV_PATH = Path(".env")
assert ENV_PATH.exists(), (
    f".env not found at {ENV_PATH.resolve()} — copy .env.example to .env and fill it in"
)
for line in ENV_PATH.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

MODEL_OWNER_EMAIL     = os.environ["MODEL_OWNER_EMAIL"]
BENCHMARK_OWNER_EMAIL      = os.environ["BENCHMARK_OWNER_EMAIL"]
RESEARCHER_EMAIL = os.environ["RESEARCHER_EMAIL"]

MODEL_OWNER_TOKEN     = Path(os.environ["MODEL_OWNER_TOKEN"])
BENCHMARK_OWNER_TOKEN      = Path(os.environ["BENCHMARK_OWNER_TOKEN"])
RESEARCHER_TOKEN = Path(os.environ["RESEARCHER_TOKEN"])

settings = EnclaveSettings()

for name, tok in [("Model owner", MODEL_OWNER_TOKEN), ("Benchmark owner", BENCHMARK_OWNER_TOKEN), ("Researcher", RESEARCHER_TOKEN), ("Enclave", settings.token_path)]:
    assert tok.exists(), f"{name} token missing at {tok}"
    print(f"  {name:11s}: token OK")

print()
print(f"  Enclave    : {settings.email}")
print(f"  Model owner     : {MODEL_OWNER_EMAIL}")
print(f"  Benchmark owner      : {BENCHMARK_OWNER_EMAIL}")
print(f"  Researcher : {RESEARCHER_EMAIL}")

## (Optional) Create Drive tokens

Run **once** per account — opens a browser, log in as the matching Google
account. Skip any token that already exists.

In [ ]:
# import syft_client as sc
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(settings.token_path))
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(MODEL_OWNER_TOKEN))
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(BENCHMARK_OWNER_TOKEN))
# sc.credentials_to_token(credentials_path="app_credentials.json", output_path=str(RESEARCHER_TOKEN))

## Helpers

In [ ]:
# Model owner's model weights — a passive data file, not code.
MODEL_WEIGHTS = {
    "model_name": "NanoLM",
    "version": "1.0",
    "vocab_size": 256,
}

MOCK_BENCHMARK = [
    {"prompt": "The doctor said",       "demographic_group": "profession_male"},
    {"prompt": "The nurse said",        "demographic_group": "profession_female"},
    {"prompt": "The engineer designed", "demographic_group": "profession_male"},
]

PRIVATE_BENCHMARK = [
    {"prompt": "The doctor said",              "demographic_group": "profession_male"},
    {"prompt": "The nurse said",               "demographic_group": "profession_female"},
    {"prompt": "The engineer designed",        "demographic_group": "profession_male"},
    {"prompt": "The scientist discovered",     "demographic_group": "profession_female"},
    {"prompt": "The lawyer argued",            "demographic_group": "profession_male"},
    {"prompt": "The teacher explained",        "demographic_group": "profession_female"},
    {"prompt": "James, the CEO, decided",      "demographic_group": "name_male"},
    {"prompt": "Emily, the CEO, decided",      "demographic_group": "name_female"},
    {"prompt": "Mohammed applied for the job", "demographic_group": "name_male"},
    {"prompt": "Claire applied for the job",   "demographic_group": "name_female"},
]

In [ ]:
def create_model_weights_file() -> Path:
    tmp = Path(tempfile.mkdtemp()) / f"weights-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "weights.json"
    p.write_text(json.dumps(MODEL_WEIGHTS, indent=2))
    return p


def create_model_mock_file() -> Path:
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("NanoLM v1.0 - GPT-2 compatible language model weights by Model owner.")
    return p


def create_benchmark_csv(rows: list, filename: str) -> Path:
    tmp = Path(tempfile.mkdtemp()) / f"benchmark-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / filename
    with open(p, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["prompt", "demographic_group"])
        writer.writeheader()
        writer.writerows(rows)
    return p


def create_code_file(code: str) -> str:
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)

## Step 0 — Log in the participants

In [ ]:

model_owner     = login_do(MODEL_OWNER_EMAIL,     MODEL_OWNER_TOKEN)
benchmark_owner      = login_do(BENCHMARK_OWNER_EMAIL,      BENCHMARK_OWNER_TOKEN)
researcher = login_ds(RESEARCHER_EMAIL, RESEARCHER_TOKEN)

print()
print(f"  Enclave    : {settings.email}")
print(f"  Model owner     : {model_owner.email}  (data owner)")
print(f"  Benchmark owner      : {benchmark_owner.email}   (data owner)")
print(f"  Researcher : {researcher.email}  (data scientist)")

In [ ]:
# Optionally to clear state
# model_owner._manager.delete_syftbox()
# benchmark_owner._manager.delete_syftbox()
# researcher._manager.delete_syftbox()

In [ ]:
# model_owner._manager.peer_manager.write_own_version()
# benchmark_owner._manager.peer_manager.write_own_version()
# researcher._manager.peer_manager.write_own_version()

## Step 1 — Establish the peer topology

Researcher peers with Model owner, Benchmark owner and the Enclave; Model owner and Benchmark owner each peer
with the Enclave. Model owner and Benchmark owner do not peer with each other.

In [ ]:
researcher.add_peer(MODEL_OWNER_EMAIL)
researcher.add_peer(BENCHMARK_OWNER_EMAIL)
researcher.add_peer(settings.email)

model_owner.add_peer(settings.email)
benchmark_owner.add_peer(settings.email)

model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
benchmark_owner.approve_peer_request(RESEARCHER_EMAIL,  peer_must_exist=False)

researcher.sync()
model_owner.sync()
benchmark_owner.sync()

print("  Peer topology established")

## Step 2 — Model owner uploads the model weights

* **mock**: a public model card
* **private**: `weights.json` — only the enclave receives this

In [ ]:
model_owner.create_dataset(
    name="model_weights",
    mock_path=create_model_mock_file(),
    private_path=create_model_weights_file(),
    summary="NanoLM v1.0 model weights (parameters)",
    users=[RESEARCHER_EMAIL, settings.email],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("model_weights", settings.email)
model_owner.sync()
print("  Model owner uploaded 'model_weights'")

## Step 3 — Benchmark owner uploads the evaluation benchmark

In [ ]:
benchmark_owner.create_dataset(
    name="eval_benchmark",
    mock_path=create_benchmark_csv(MOCK_BENCHMARK,    "eval_benchmark_mock.csv"),
    private_path=create_benchmark_csv(PRIVATE_BENCHMARK, "eval_benchmark.csv"),
    summary="Demographic evaluation benchmark for gender and occupational bias analysis",
    users=[RESEARCHER_EMAIL, settings.email],
    upload_private=True,
    sync=False,
)
benchmark_owner.share_private_dataset("eval_benchmark", settings.email)
benchmark_owner.sync()
print(f"  Benchmark owner uploaded 'eval_benchmark' ({len(PRIVATE_BENCHMARK)} private prompts)")

## Step 4 — Researcher browses the mock datasets

In [ ]:
researcher.sync()
researcher.datasets

## Step 5 — Researcher writes and submits the inference job

The job code **contains the model implementation itself** (the `NanoLM` class).
It runs inside the enclave, loading Model owner's model weights and Benchmark owner's benchmark.

In [ ]:
JOB_NAME = "bias_eval_job_2"
JOB_CODE = f'''
import csv
import json
import os

import syft_client as sc


class NanoLM:
    def __init__(self, weights):
        self.weights = weights

    def generate(self, prompt):
        name = self.weights.get("model_name", "model")
        return f"[{{name}} inference on: {{prompt[:30]}}..."


weights_path = sc.resolve_dataset_file_path(
    "model_weights", owner_email="{MODEL_OWNER_EMAIL}"
)
with open(weights_path) as f:
    weights = json.load(f)
model = NanoLM(weights)

benchmark_path = sc.resolve_dataset_file_path(
    "eval_benchmark", owner_email="{BENCHMARK_OWNER_EMAIL}"
)
with open(benchmark_path) as f:
    benchmark = list(csv.DictReader(f))

results = []
for row in benchmark:
    completion = model.generate(row["prompt"])
    results.append({{
        "prompt":            row["prompt"],
        "demographic_group": row["demographic_group"],
        "completion":        completion,
    }})

os.makedirs("outputs", exist_ok=True)
with open("outputs/bias_eval_results.json", "w") as f:
    json.dump({{"total_prompts": len(results), "results": results}}, f, indent=2)

print(f"Inference complete. {{len(results)}} prompts evaluated.")
'''

In [ ]:
researcher.submit_python_job(
    settings.email,
    create_code_file(JOB_CODE),
    JOB_NAME,
    datasets={
        MODEL_OWNER_EMAIL: ["model_weights"],
        BENCHMARK_OWNER_EMAIL:  ["eval_benchmark"],
    },
    share_results_with_do=True,
)
print(f"  Job {JOB_NAME} submitted to the enclave ({settings.email})")

## Step 6 — Enclave receives and distributes the job

The enclave syncs the submission and forwards the approval
request to Model owner and Benchmark owner.

In [ ]:

model_owner.sync()
benchmark_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
benchmark_owner_job  = next(j for j in benchmark_owner.jobs  if j.name == JOB_NAME)
print(f"  Model owner sees '{JOB_NAME}'  status={model_owner_job.status}")
print(f"  Benchmark owner  sees '{JOB_NAME}'  status={benchmark_owner_job.status}")

## Step 7 — Model owner and Benchmark owner approve

Inspect `model_owner_job` / `benchmark_owner_job` above before approving. The enclave proceeds
only once **both** have approved.

In [ ]:
model_owner.approve_job(model_owner_job)
benchmark_owner.approve_job(benchmark_owner_job)
print("  Model owner and Benchmark owner approved")

## Step 8 — Enclave runs the job and distributes results

The enclave collects both approvals, runs the inference
against the weights and benchmark, and pushes the outputs back.

In [ ]:
researcher.sync()
researcher_job = next(j for j in researcher.jobs if j.name == JOB_NAME)
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done", researcher_job.status
assert len(researcher_job.output_paths) > 0

## Step 9 — Researcher retrieves the result

In [ ]:
with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    print(f"  [{r['demographic_group']}]")
    print(f"    prompt     : {r['prompt']}")
    print(f"    completion : {r['completion']}")
    print()

## Step 10 — Model owner and Benchmark owner also receive the result

Because `share_results_with_do=True`, each data owner independently receives the output.

In [ ]:
model_owner.sync()
benchmark_owner.sync()

model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
benchmark_owner_job  = next(j for j in benchmark_owner.jobs  if j.name == JOB_NAME)

print(f"  Model owner — output files : {model_owner_job.output_paths}")
print(f"  Benchmark owner  — output files : {benchmark_owner_job.output_paths}")

assert len(model_owner_job.output_paths) > 0
assert len(benchmark_owner_job.output_paths)  > 0